# Treatment deserts: which U.S. counties are farthest from detox and methadone?

Quick look at the [US County Distance to Addiction Treatment](https://www.kaggle.com/datasets/nick722/us-county-distance-to-addiction-treatment) dataset. Distances are straight-line miles from each county's 2020 population center. Interactive map and county pages: [findrehabmap.com](https://findrehabmap.com).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import glob
path = glob.glob('/kaggle/input/**/us-county-treatment-access.csv', recursive=True)[0]
df = pd.read_csv(path, dtype={'fips': str})
st = pd.read_csv(path.replace('us-county-treatment-access.csv', 'state-summary.csv'))
df.shape

## How many people live 30+ miles from medical detox?

In [ ]:
far = df[df.miles_to_nearest_detox > 30]
print(f'{len(far):,} counties, {far.population_2024.sum()/1e6:.1f} million people')
print(f'{(df.facilities_in_county == 0).sum():,} counties have no treatment program inside the county')

## States ranked by share of residents far from detox

In [ ]:
st[['rank','state','share_30mi_detox','people_30mi_detox','counties_no_program','counties']].head(15)

## Larger counties (50k+ people) farthest from a methadone clinic

In [ ]:
df[df.population_2024 >= 50000].nlargest(15, 'miles_to_nearest_methadone')[['county','state','population_2024','miles_to_nearest_methadone','nearest_methadone_city']]

## Distance to detox vs. household income

In [ ]:
d = df.dropna(subset=['median_household_income','miles_to_nearest_detox'])
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(d.median_household_income, d.miles_to_nearest_detox, s=6, alpha=.35)
ax.set_xlabel('Median household income (USD)'); ax.set_ylabel('Miles to nearest detox'); ax.set_ylim(0, 200)
ax.set_title('County income vs. distance to medical detox')
plt.show()
print('Spearman rho:', round(d[['median_household_income','miles_to_nearest_detox']].corr(method='spearman').iloc[0,1], 3))

Straight-line distance understates the real trip, especially in mountain West counties. Buprenorphine access is also under-counted because many primary care doctors and telehealth services prescribe it without appearing in SAMHSA's locator. Method and cleaning steps: [findrehabmap.com/about](https://findrehabmap.com/about/).